# 跨城市 x 7情景 分组柱状图（扩容 + 定容 各一张）\n\n数据组合方式（与原 5.py 一致）：\n- **2020 Baseline** 取自「现状」策略\n- **2040 / 2060 的 RCP 各情景** 分别取自「扩容」和「定容」策略，各画一张图\n\n自动处理 26°C 和 27°C 两个温度基准，各生成一套独立的图表。\n26°C 数据可能只覆盖部分城市，代码自动适配。

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from config.paths import get_per_capita_output_dir, get_figures_dir
from config.parameters import BASELINES
from src.plotting.grouped_bar import plot_scenario_grouped

In [ ]:
def build_plot_df(df, strategy_name):
    """组合：现状的 2020 Baseline + 指定策略的所有 RCP 情景"""
    df_baseline = df[(df['策略'] == '现状') & (df['情景'] == '2020 Baseline')]
    df_future = df[(df['策略'] == strategy_name) & (df['情景'] != '2020 Baseline')]
    return pd.concat([df_baseline, df_future], ignore_index=True)


for baseline in BASELINES:
    csv_path = os.path.join(get_per_capita_output_dir(baseline), 'per_capita_hours_summary.csv')
    if not os.path.exists(csv_path):
        print(f'!! [{baseline}°C] 数据文件不存在，跳过: {csv_path}')
        continue

    df = pd.read_csv(csv_path)
    cities = sorted(df['城市'].unique())
    print(f'[{baseline}°C] 覆盖城市: {cities}')
    print(f'[{baseline}°C] 共 {len(df)} 行数据')

    fig_dir = get_figures_dir(baseline)
    os.makedirs(fig_dir, exist_ok=True)

    # 扩容版
    df_exp = build_plot_df(df, '扩容')
    save_path = os.path.join(fig_dir, f'Grouped_BarChart_扩容_{baseline}Degree_SCI_600DPI.png')
    fig1, ax1 = plot_scenario_grouped(df_exp, save_path=save_path, add_grand_total=False)
    print(f'   [OK] 扩容版 → {save_path}')

    # 定容版
    df_fix = build_plot_df(df, '定容')
    save_path = os.path.join(fig_dir, f'Grouped_BarChart_定容_{baseline}Degree_SCI_600DPI.png')
    fig2, ax2 = plot_scenario_grouped(df_fix, save_path=save_path, add_grand_total=False)
    print(f'   [OK] 定容版 → {save_path}')

print('\n>> 全部图表生成完成！')